
# Project Report: Top-View Person Re-Identification (TVRID)

**Course Name / Assignment:** Computer vision
**Student Names :** Sparsh Rastogi (2301ai56)  , Abhay Pratap Singh (2301ai48) , Atul Raj Chaudhary (2301ai40)
**Date:** 18 March 2026

---

## 1. Abstract
Person Re-Identification (Re-ID) is a critical task in computer vision, focusing on matching individuals across non-overlapping camera views. This project tackles the specific and challenging domain of **Top-View Person Re-Identification**, where facial features are heavily occluded, and the system must rely on features like clothing, shoulder contours, and depth profiles. We developed a multi-modal (RGB and Depth) Re-ID system using PyTorch Lightning. We evaluated multiple backbone architectures (OSNet, ResNet50, ConvNeXt-T, and ViT-B/16) and compared Metric Learning (Triplet Loss) against Classification Learning (ID / Cross-Entropy Loss). Our results demonstrate that a Vision Transformer (ViT-B/16) trained with ID Loss on the RGB track achieves a state-of-the-art mean Average Precision (mAP) of 0.95, while the Depth track utilizing ResNet50 achieves an impressive mAP of 0.96.

## 2. Introduction
Traditional Re-ID systems rely heavily on frontal or profile views of individuals. However, in retail, surveillance, and robotics, cameras are frequently mounted directly overhead (Top-View). This perspective removes traditional discriminative features (like faces and chest logos) and introduces severe variations in appearance due to pose changes.

To address these challenges, this project investigates:
1.  **Multi-Modality:** Using both standard visual spectrum cameras (RGB) and structural geometry cameras (Depth).
2.  **Architectural Impact:** Comparing lightweight Re-ID specific models (OSNet) against heavy generalized Convolutional Neural Networks (ResNet, ConvNeXt) and Vision Transformers (ViT).
3.  **Optimization Strategies:** Analyzing the impact of relative comparison (Triplet Loss) versus absolute categorization (ID Loss) on a dataset of 88 unique identities.

## 3. Methodology

### 3.1 Dataset and Preprocessing
The model was trained on the TVRID (Top-View Re-ID) dataset, consisting of 88 unique identities. 
To prevent overfitting and simulate real-world camera noise, we applied robust data augmentation pipelines:
* **Geometric:** Random Horizontal Flips (50% probability) and Random Cropping with padding.
* **Occlusion:** Random Erasing (replacing 2% to 15% of the image with random noise) to simulate occlusions.
* **Standardization:** All images were resized to $224 \times 224$ pixels to satisfy the strict grid requirements of Vision Transformers while maintaining consistency across CNNs.

### 3.2 Architectures
We engineered a modular system to test four distinct feature extractors:
1.  **OSNet (Omni-Scale Network):** A lightweight network (2.5M parameters) specifically designed for Re-ID by fusing multi-scale features.
2.  **ResNet50:** A standard, robust 25M parameter CNN. To manage GPU memory constraints, the early generic feature layers (Layers 1-3) were frozen, fine-tuning only the highest semantic block (Layer 4).
3.  **ConvNeXt-Tiny:** A modern CNN (28M parameters) modernized with Transformer-like design choices.
4.  **ViT-B/16:** A Vision Transformer that processes the image as a sequence of $16 \times 16$ patches, excellent for capturing global context.

### 3.3 Loss Formulations
We experimented with two primary training objectives:
* **Triplet Loss (Metric Learning):** The model processes an Anchor, a Positive (same person), and a Negative (different person) image, penalizing the network if the Anchor is closer to the Negative than the Positive by a set margin.
* **ID Loss (Cross-Entropy Classification):** The Re-ID problem is reframed as an 88-class image classification task. A fully connected classification head predicts the identity. During evaluation, this head is discarded, and the penultimate 128-dimensional embedding is used as the feature descriptor.

### 3.4 Evaluation Strategy
For inference, the classification head was removed. We embedded all Query and Gallery images using the trained backbones. A **K-Nearest Neighbors (KNN)** retrieval approach was employed, calculating the Euclidean distance (`torch.cdist`) between the query and all gallery images. The primary metrics used for evaluation are **mean Average Precision (mAP)**, **Rank-1**, and **Rank-5** accuracy.

---

## 4. Results

The table below summarizes the evaluation of our models on the test set across different tracks, backbones, and loss functions.

| Track | Backbone | Image Size | Loss Formulation | mAP | Rank-1 | Rank-5 |
| :--- | :--- | :---: | :--- | :---: | :---: | :---: |
| **RGB** | OSNet | 224x224 | Triplet Loss | 0.88 | 0.82 | 1.00 |
| **RGB** | ResNet50 | 224x224 | ID (CrossEntropy) | 0.91 | 0.94 | 1.00 |
| **RGB** | ConvNeXt-T | 224x224 | ID (CrossEntropy) | 0.92 | 0.96 | 1.00 |
| **RGB** | ViT-B/16 | 224x224 | ID (CrossEntropy) | **0.95** | **0.98** | **1.00** |
| | | | | | | |
| **Depth**| OSNet | 224x224 | Triplet Loss | 0.94 | 0.97 | 1.00 |
| **Depth**| ResNet50 | 224x224 | ID (CrossEntropy) | **0.96** | **0.97** | **1.00** |

---

## 5. Analysis and Discussion

### 5.1 Superiority of ID Loss over Triplet Loss
The results clearly indicate that ID Loss (Classification) outperforms pure Triplet Loss on this dataset. Because the dataset is relatively small (88 identities), framing the objective as an absolute classification task forces the network to memorize distinct, highly discriminative features for every individual. When the classification head is removed, these rich embeddings translate to vastly superior retrieval performance (e.g., RGB ResNet50 achieved 0.91 mAP compared to OSNet's 0.88 mAP).

### 5.2 Transformer vs. CNN Architectures
In the RGB track, the progression of architectures shows a clear trend. The Vision Transformer (ViT-B/16) achieved the highest performance (mAP: 0.95, Rank-1: 0.98). ViT's self-attention mechanism allows it to holistically relate different patches of the top-view image (e.g., correlating the color of a shoe with the pattern on a shoulder) without the strict inductive bias of convolutions. ConvNeXt-T closely followed (mAP: 0.92), proving that modernized CNNs are highly competitive.

### 5.3 Modality: Depth is Highly Discriminative
Interestingly, the Depth modality proved to be inherently robust for Top-View Re-ID. Even with the lighter OSNet and Triplet Loss, the Depth track achieved an mAP of 0.94 and Rank-1 of 0.97, massively outperforming the equivalent RGB setup. When upgraded to ResNet50 with ID Loss, Depth mAP reached 0.96. This is because Depth images are unaffected by lighting changes, shadows, or similar clothing colors, relying purely on the structural geometry (height and shoulder width) of the individuals.

### 5.4 Rank-5 Convergence
Across all experiments, the Rank-5 accuracy converged to a perfect 1.00 (100%). This indicates that regardless of the backbone or modality, the system is highly reliable at narrowing down the identity; the correct individual is *always* present within the top 5 retrieved candidates.

## 6. Conclusion
In this project, we successfully built and evaluated a Top-View Person Re-Identification system. We demonstrated that for closed-set, mid-sized datasets, utilizing an ID/Cross-Entropy classification bottleneck yields highly robust feature embeddings. Furthermore, Vision Transformers (ViT) are exceptionally well-suited for RGB feature extraction in unusual camera perspectives. Finally, our experiments validate that Depth information provides an incredibly stable structural cue for identity matching, practically rivaling or exceeding the performance of high-resolution RGB data.